# replace-final-head — ex2: freeze the backbone after swapping the head and verify only the head has gradients

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `replace-final-head`. Running the final beacon cell reports progress against the `Transfer: Replace final head` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
import torch.nn as nn
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Transfer: Replace final head` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`replace-final-head`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "replace-final-head"
DD_SUBTOPIC = "Transfer: Replace final head"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Freeze the backbone, train only the new head

Ex1 SWAPPED the final classifier. The deepening move is what almost always comes next in transfer learning: freeze the backbone so the optimizer only updates the new head's parameters.

```python
model.fc = nn.Linear(in_features, new_num_classes)
for p in model.backbone.parameters():
    p.requires_grad_(False)
```

**`requires_grad_(False)` is in-place.** The trailing underscore mutates the tensor's flag without returning a new one. Setting `p.requires_grad = False` directly also works, but the trailing-underscore form is the canonical PyTorch idiom for in-place flag updates.

**Why the new head is automatically trainable.** Brand-new `nn.Linear(...)` parameters default to `requires_grad=True`. You only have to flip the OLD layers off — the new layer is already on.

**Verifying the freeze worked.** After a `loss.backward()`, only the head params should have non-`None` `.grad`. The backbone params have `.grad is None` because no graph was built for them.

### Exercise 2 — freeze the backbone after swapping the head and verify only the head has gradients

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a head-swapped model post-`backward` to confirm that after freezing the backbone with `requires_grad_(False)`, only the new head's parameters carry gradients (`.grad is not None`) while the backbone params have `.grad is None`.
> Keywords: transfer-learning, freeze, requires_grad, backward
> ```

**KCs targeted:** `requires-grad-false-in-place`, `verify-grad-only-on-new-head`

Implement `ex2_swap_freeze_and_check(model, new_num_classes, x, y)`.

Steps:
1. Read `in_features = model.fc.in_features`.
2. Replace `model.fc = nn.Linear(in_features, new_num_classes)` — the new head is automatically trainable.
3. Freeze the backbone: for every parameter of `model.backbone`, call `p.requires_grad_(False)`.
4. Forward `out = model(x)` then compute `loss = nn.functional.cross_entropy(out, y)`.
5. Call `loss.backward()`.
6. Return a dict:
   ```
   {
     'head_param_names_with_grad': [names with .grad is not None],
     'backbone_param_names_with_grad': [names with .grad is not None],
     'head_grad_norms': {name: float(p.grad.norm())} for those params,
   }
   ```
   Use `model.named_parameters()` and split on the qname prefix (`'fc.'` for head, `'backbone.'` for backbone).

Expected outcome (drives the test):
- `head_param_names_with_grad` is non-empty (head trained).
- `backbone_param_names_with_grad` is empty (backbone frozen).
- Every head grad norm is strictly positive.

In [ ]:
def ex2_swap_freeze_and_check(model, new_num_classes, x, y):
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, new_num_classes)
    for p in model.backbone.parameters():
        p.requires_grad_(False)
    out = model(x)
    loss = nn.functional.cross_entropy(out, y)
    loss.backward()
    head_with_grad = []
    backbone_with_grad = []
    head_norms = {}
    for name, p in model.named_parameters():
        if name.startswith('fc.'):
            if p.grad is not None:
                head_with_grad.append(name)
                head_norms[name] = float(p.grad.norm())
        elif name.startswith('backbone.'):
            if p.grad is not None:
                backbone_with_grad.append(name)
    return {
        'head_param_names_with_grad': sorted(head_with_grad),
        'backbone_param_names_with_grad': sorted(backbone_with_grad),
        'head_grad_norms': head_norms,
    }


<details><summary>Solution</summary>

```python
def ex2_swap_freeze_and_check(model, new_num_classes, x, y):
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, new_num_classes)
    for p in model.backbone.parameters():
        p.requires_grad_(False)
    out = model(x)
    loss = nn.functional.cross_entropy(out, y)
    loss.backward()
    head_with_grad = []
    backbone_with_grad = []
    head_norms = {}
    for name, p in model.named_parameters():
        if name.startswith('fc.'):
            if p.grad is not None:
                head_with_grad.append(name)
                head_norms[name] = float(p.grad.norm())
        elif name.startswith('backbone.'):
            if p.grad is not None:
                backbone_with_grad.append(name)
    return {
        'head_param_names_with_grad': sorted(head_with_grad),
        'backbone_param_names_with_grad': sorted(backbone_with_grad),
        'head_grad_norms': head_norms,
    }
```

**`p.grad is None` is the freeze tell, not `p.grad == 0`.** When a parameter has `requires_grad=False`, autograd never builds a graph node for it, so `.grad` stays at its initial `None`. A frozen param with grad zero would mean the graph WAS built but the gradient happened to vanish — different bug.

**`requires_grad_(False)` recurses through `parameters()`.** `model.backbone.parameters()` walks all leaves under backbone, regardless of nesting depth. The trailing-underscore form is the in-place idiom; you could also do `p.requires_grad = False` — same effect.

**Fresh `nn.Linear` is trainable by default.** No need to set `requires_grad=True` after the swap — `nn.Parameter` defaults to `True`. Only the OLD layers needed flipping.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()